# ENSO RL — Analysis Notebook

All figures and tables for the paper. Run computation scripts first to generate `.npz`/`.csv` files, then use this notebook to produce publication-quality plots without recomputing.

**Computation scripts** (run from repo root):
```bash
uv run scripts/analysis/lift_analysis.py          --model ensemble
uv run scripts/analysis/counterfactual_analysis.py --model ensemble
uv run scripts/analysis/shapley_analysis.py        --model ensemble
uv run scripts/analysis/interventional_xro.py      --model ensemble
uv run scripts/analysis/seasonality_of_control.py  --model ensemble
uv run scripts/analysis/precursor_composite.py     --model ensemble
uv run scripts/analysis/mutual_information.py      --model ensemble
uv run scripts/analysis/integrated_gradients.py   --model ensemble
uv run scripts/analysis/gradient_sensitivity.py   --model ensemble
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from scipy.stats import spearmanr, ttest_1samp

NAME = "ensemble"   # matches --model used when running computation scripts
BASE = Path("..") / "plots" / NAME

PHASES       = ["total", "el_nino", "la_nina"]
PHASE_LABELS = {"total": "Total MYE", "el_nino": "Multi-year El Niño", "la_nina": "Multi-year La Niña"}
PHASE_COLORS = {"total": "#4878CF", "el_nino": "#D65F5F", "la_nina": "#5CB85C"}
XRO_MODES    = ["WWV", "NPMM", "SPMM", "IOB", "IOD", "SIOD", "TNA", "ATL3", "SASD"]

def _zscore(x):
    x = np.asarray(x, float)
    sd = x.std()
    return (x - x.mean()) / sd if sd > 1e-12 else x - x.mean()

def _sig_stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

def _add_sig_labels(ax, x_positions, values, pvals, offset_frac=0.04):
    """Annotate bar chart with significance stars."""
    ylim = ax.get_ylim()
    offset = (ylim[1] - ylim[0]) * offset_frac
    for xi, val, p in zip(x_positions, values, pvals):
        star = _sig_stars(p)
        y = val + offset if val >= 0 else val - offset
        va = "bottom" if val >= 0 else "top"
        ax.text(xi, y, star, ha="center", va=va, fontsize=9, fontweight="bold")

def _load_npz(path):
    """Load npz and return as a dict-like NpzFile (supports 'key in d.files')."""
    return np.load(path, allow_pickle=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
print(f"BASE = {BASE.resolve()}")

---
## 1. Lift Analysis

ΔP(MYE) = agent probability − baseline probability, decomposed into frequency and duration changes.

In [ ]:
lift_dir = BASE / "lift"
d_lift   = _load_npz(lift_dir / "lift_ensemble.npz")
df_lift  = pd.read_csv(lift_dir / "lift_ensemble.csv")
df_decomp     = pd.read_csv(lift_dir / "lift_decomposition.csv")
df_decomp_ps  = pd.read_csv(lift_dir / "lift_decomposition_per_seed.csv")

print("Lift summary:")
print(df_lift.to_string(index=False))

In [ ]:
# --- 1a. Lift bar chart per phase ---
fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(PHASES))
labels = [PHASE_LABELS[p] for p in PHASES]

lifts  = df_lift.set_index("phase").loc[PHASES, "lift"].values
ci95   = df_lift.set_index("phase").loc[PHASES, "lift_ci95"].values
pvals  = df_lift.set_index("phase").loc[PHASES, "p"].values if "p" in df_lift.columns else np.ones(3)
colors = [PHASE_COLORS[p] for p in PHASES]

bars = ax.bar(x, lifts, color=colors, edgecolor="black", linewidth=1.2,
              yerr=ci95, capsize=6, error_kw={"linewidth": 1.5})
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
_add_sig_labels(ax, x, lifts, pvals)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("ΔP(MYE)  [agent − baseline]")
ax.set_title("MYE Lift by Phase")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# --- 1b. Frequency & duration decomposition ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(PHASES))
labels = [PHASE_LABELS[p] for p in PHASES]
dc = df_decomp.set_index("phase").loc[PHASES]

# Frequency
ax = axes[0]
w = 0.35
ax.bar(x - w/2, dc["freq_agent_per100yr"], w, label="Agent", color="#4878CF",
       yerr=dc["freq_agent_ci95"], capsize=5, error_kw={"linewidth": 1.2}, edgecolor="black")
ax.bar(x + w/2, dc["freq_base_per100yr"],  w, label="Baseline", color="#aaaaaa",
       yerr=dc["freq_base_ci95"],  capsize=5, error_kw={"linewidth": 1.2}, edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Events per 100 years")
ax.set_title("MYE Frequency")
ax.legend(); ax.grid(axis="y", alpha=0.3)

# Duration
ax = axes[1]
ax.bar(x - w/2, dc["dur_agent_mo"], w, label="Agent", color="#4878CF",
       yerr=dc["dur_agent_ci95"], capsize=5, error_kw={"linewidth": 1.2}, edgecolor="black")
ax.bar(x + w/2, dc["dur_base_mo"],  w, label="Baseline", color="#aaaaaa",
       yerr=dc["dur_base_ci95"],  capsize=5, error_kw={"linewidth": 1.2}, edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Mean event duration (months)")
ax.set_title("MYE Duration")
ax.legend(); ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# --- 1c. Per-seed freq–duration scatter (robustness check) ---
fig, axes = plt.subplots(1, len(PHASES), figsize=(14, 4), sharey=False)
for ax, phase in zip(axes, PHASES):
    sub = df_decomp_ps[df_decomp_ps["phase"] == phase]
    ax.scatter(sub["freq_agent_per100yr"], sub["dur_agent_mo"],
               color=PHASE_COLORS[phase], s=80, zorder=3, label="Agent")
    ax.scatter(sub["freq_base_per100yr"],  sub["dur_base_mo"],
               color="#aaaaaa", marker="x", s=80, zorder=3, label="Baseline")
    ax.set_xlabel("Frequency (events/100yr)")
    ax.set_ylabel("Duration (months)")
    ax.set_title(PHASE_LABELS[phase])
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.suptitle("Per-seed Frequency vs Duration", y=1.02)
fig.tight_layout()
plt.show()

---
## 2. Counterfactual Analysis

Zero-ablation of each action dimension. ΔP(MYE) measures how much removing a mode degrades (or improves) the agent's ability to sustain multi-year ENSO.

In [ ]:
cf_dir = BASE / "counterfactual"
d_cf   = _load_npz(cf_dir / "counterfactual_ensemble.npz")

features = list(d_cf["features"])  # ['WWV', 'NPMM', ...]

# Build p-values: prefer stored p_* keys; fall back to computing from per_seed
cf_pvals = {}
for p in PHASES:
    if f"p_{p}" in d_cf.files:
        cf_pvals[p] = d_cf[f"p_{p}"]
    elif f"per_seed_{p}" in d_cf.files:
        M = d_cf[f"per_seed_{p}"]  # [n_seeds, n_features]
        cf_pvals[p] = np.array([ttest_1samp(M[:, fi], 0).pvalue for fi in range(M.shape[1])])
    else:
        cf_pvals[p] = np.ones(len(features))

print(f"Features: {features}")
print(f"Phases: {list(d_cf['phases'])}")

In [ ]:
# --- 2. Counterfactual ΔP bars per phase ---
fig, axes = plt.subplots(1, len(PHASES), figsize=(15, 5), sharey=False)
for ax, phase in zip(axes, PHASES):
    means = d_cf[f"mean_{phase}"]
    cis   = d_cf[f"ci_{phase}"]
    pvals = cf_pvals[phase]

    order = np.argsort(means)[::-1]
    names_s = [features[i] for i in order]
    means_s = means[order]
    cis_s   = cis[order]
    pvals_s = pvals[order]

    colors = [PHASE_COLORS[phase] if p < 0.05 else "#cccccc" for p in pvals_s]
    x = np.arange(len(features))
    ax.bar(x, means_s, color=colors, edgecolor="black", linewidth=1.0,
           yerr=cis_s, capsize=5, error_kw={"linewidth": 1.2})
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    _add_sig_labels(ax, x, means_s, pvals_s)
    ax.set_xticks(x); ax.set_xticklabels(names_s, rotation=45, ha="right")
    ax.set_ylabel("ΔP(MYE) after zero-ablation")
    ax.set_title(PHASE_LABELS[phase])
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Counterfactual Analysis — Driver Importance", fontsize=13)
fig.tight_layout()
plt.show()

---
## 3. Shapley Analysis

Permutation-based Shapley values across seeds — a game-theoretic attribution of each mode's marginal contribution to the agent's MYE lift.

In [ ]:
sh_dir = BASE / "shapley"
d_sh   = _load_npz(sh_dir / "shapley_ensemble.npz")

sh_features = list(d_sh["features"])

sh_pvals = {}
for p in PHASES:
    if f"p_{p}" in d_sh.files:
        sh_pvals[p] = d_sh[f"p_{p}"]
    elif f"per_seed_{p}" in d_sh.files:
        M = d_sh[f"per_seed_{p}"]
        sh_pvals[p] = np.array([ttest_1samp(M[:, fi], 0).pvalue for fi in range(M.shape[1])])
    else:
        sh_pvals[p] = np.ones(len(sh_features))

print(f"Features: {sh_features}")

In [ ]:
# --- 3a. Shapley bars per phase ---
fig, axes = plt.subplots(1, len(PHASES), figsize=(15, 5), sharey=False)
for ax, phase in zip(axes, PHASES):
    means = d_sh[f"mean_{phase}"]
    cis   = d_sh[f"ci_{phase}"]
    pvals = sh_pvals[phase]

    order = np.argsort(means)[::-1]
    names_s = [sh_features[i] for i in order]
    means_s = means[order]
    cis_s   = cis[order]
    pvals_s = pvals[order]

    colors = [PHASE_COLORS[phase] if p < 0.05 else "#cccccc" for p in pvals_s]
    x = np.arange(len(sh_features))
    ax.bar(x, means_s, color=colors, edgecolor="black", linewidth=1.0,
           yerr=cis_s, capsize=5, error_kw={"linewidth": 1.2})
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    _add_sig_labels(ax, x, means_s, pvals_s)
    ax.set_xticks(x); ax.set_xticklabels(names_s, rotation=45, ha="right")
    ax.set_ylabel("Shapley value (ΔP contribution)")
    ax.set_title(PHASE_LABELS[phase])
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Shapley Attribution — Driver Importance", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# --- 3b. Per-seed Shapley heatmap (if per_seed_* arrays available) ---
if "per_seed_total" in d_sh.files:
    M = d_sh["per_seed_total"]  # [n_seeds, n_features]
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.imshow(M.T, cmap="RdBu_r", aspect="auto",
                   norm=mcolors.TwoSlopeNorm(vcenter=0))
    ax.set_yticks(range(len(sh_features))); ax.set_yticklabels(sh_features)
    ax.set_xlabel("Seed index")
    ax.set_title("Shapley values per seed (Total MYE)")
    plt.colorbar(im, ax=ax, label="Shapley value")
    fig.tight_layout()
    plt.show()
else:
    print("per_seed_* arrays not in shapley npz — re-run shapley_analysis.py to populate.")

---
## 4. Driver Convergence

Z-scored overlay of counterfactual, Shapley, and interventional methods. When all three agree, the driver ranking is robust. Spearman ρ quantifies cross-method consistency.

In [ ]:
d_iv = _load_npz(BASE / "interventional_xro" / "interventional_xro.npz")

iv_targets = list(d_iv["targets"])    # e.g. 'WWV_+', 'WWV_-', ...
iv_phases  = list(d_iv["phases"])
iv_means   = d_iv["mean"]
iv_signs   = list(d_iv["signs"])

def _iv_for_phase(phase, features):
    """Return best (max absolute) ΔP per feature for a given phase from interventional."""
    vals = np.zeros(len(features))
    for i, feat in enumerate(features):
        idxs = [j for j, (t, ph) in enumerate(zip(iv_targets, iv_phases))
                if ph == phase and t.startswith(feat)]
        if idxs:
            vals[i] = iv_means[idxs][np.argmax(np.abs(iv_means[idxs]))]
    return vals

print(f"Interventional: {len(iv_targets)} target×phase entries")

In [ ]:
# --- 4. Z-scored driver convergence overlay ---
fig, axes = plt.subplots(1, len(PHASES), figsize=(15, 5), sharey=False)
x = np.arange(len(features))
w = 0.25

for ax, phase in zip(axes, PHASES):
    cf_z  = _zscore(d_cf[f"mean_{phase}"])
    sh_z  = _zscore(d_sh[f"mean_{phase}"])
    iv_z  = _zscore(_iv_for_phase(phase, features))

    ax.bar(x - w, cf_z, w, label="Counterfactual", color="#4878CF", edgecolor="black", linewidth=0.8)
    ax.bar(x,     sh_z, w, label="Shapley",         color="#D65F5F", edgecolor="black", linewidth=0.8)
    ax.bar(x + w, iv_z, w, label="Interventional",  color="#5CB85C", edgecolor="black", linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_xticks(x); ax.set_xticklabels(features, rotation=45, ha="right")
    ax.set_ylabel("Z-score")
    ax.set_title(PHASE_LABELS[phase])
    ax.grid(axis="y", alpha=0.3)

    # Spearman ρ
    rho_cf_sh, _ = spearmanr(cf_z, sh_z)
    rho_cf_iv, _ = spearmanr(cf_z, iv_z)
    rho_sh_iv, _ = spearmanr(sh_z, iv_z)
    ax.set_xlabel(f"ρ(CF,Sh)={rho_cf_sh:.2f}  ρ(CF,IV)={rho_cf_iv:.2f}  ρ(Sh,IV)={rho_sh_iv:.2f}",
                  fontsize=8)

axes[0].legend(fontsize=9)
fig.suptitle("Driver Convergence — Z-scored Cross-Method Comparison", fontsize=13)
fig.tight_layout()
plt.show()

---
## 5. Seed Sensitivity

Robustness of results across independently trained seeds. We check (a) whether the El Niño vs La Niña lift asymmetry is consistent per seed, and (b) whether driver rankings from counterfactual and Shapley are stable across seeds.

In [ ]:
# Load per-seed lift arrays
lift_en = d_lift["per_seed_lift_el_nino"]  # [n_seeds]
lift_ln = d_lift["per_seed_lift_la_nina"]  # [n_seeds]
n_seeds = len(lift_en)
print(f"Seeds: {n_seeds}  |  El Niño lift: {lift_en.mean():.3f} ± {lift_en.std():.3f}")
print(f"                    La Niña lift: {lift_ln.mean():.3f} ± {lift_ln.std():.3f}")

In [ ]:
# --- 5a. EN vs LN lift scatter per seed ---
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(lift_en, lift_ln, s=80, color="#4878CF", edgecolor="black", zorder=3)
for i, (x_, y_) in enumerate(zip(lift_en, lift_ln)):
    ax.text(x_ + 0.005, y_ + 0.005, str(i), fontsize=8)
lims = [min(lift_en.min(), lift_ln.min()) - 0.05,
        max(lift_en.max(), lift_ln.max()) + 0.05]
ax.plot(lims, lims, "k--", linewidth=0.8, label="EN = LN")
ax.axhline(0, color="grey", linewidth=0.6); ax.axvline(0, color="grey", linewidth=0.6)
ax.set_xlabel("Multi-year El Niño lift"); ax.set_ylabel("Multi-year La Niña lift")
ax.set_title("Per-seed EN vs LN Lift Asymmetry")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# --- 5b. Seed × mode heatmaps (CF and Shapley, total phase) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (d, label) in zip(axes, [(d_cf, "Counterfactual"), (d_sh, "Shapley")]):
    if "per_seed_total" in d.files:
        M = d["per_seed_total"]   # [n_seeds, n_features]
        feat = list(d["features"])
        im = ax.imshow(M, cmap="RdBu_r", aspect="auto",
                       norm=mcolors.TwoSlopeNorm(vcenter=0))
        ax.set_xticks(range(len(feat))); ax.set_xticklabels(feat, rotation=45, ha="right")
        ax.set_yticks(range(M.shape[0])); ax.set_yticklabels([f"seed {i}" for i in range(M.shape[0])])
        ax.set_title(f"{label} — per seed (Total MYE)")
        plt.colorbar(im, ax=ax, label="ΔP")
    else:
        ax.text(0.5, 0.5, f"per_seed_total\nnot in {label} npz\n(re-run script)",
                ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_title(f"{label} — per seed (Total MYE)")

fig.tight_layout()
plt.show()

---
## 6. Interventional XRO

Direct mode-press and mode-brake interventions in a free XRO. Shows which modes, when externally sustained or suppressed, causally change MYE probability.

In [ ]:
# interventional_xro.npz keys: targets, signs, phases, mean, ci, p
iv_df = pd.DataFrame({
    "target": list(d_iv["targets"]),
    "sign":   list(d_iv["signs"]),
    "phase":  list(d_iv["phases"]),
    "mean":   d_iv["mean"],
    "ci":     d_iv["ci"],
    "p":      d_iv["p"] if "p" in d_iv.files else np.ones(len(d_iv["targets"])),
})
# Extract mode name (everything before last _)
iv_df["mode"] = iv_df["target"].str.rsplit("_", n=1).str[0]
print(iv_df.head(6).to_string())

In [ ]:
# --- 6. Interventional ΔP bars per phase ---
fig, axes = plt.subplots(1, len(PHASES), figsize=(15, 5), sharey=False)
for ax, phase in zip(axes, PHASES):
    sub = iv_df[iv_df["phase"] == phase].copy()
    sub = sub.sort_values("mean", ascending=False)

    colors = ["#D65F5F" if s == "+" else "#4878CF" for s in sub["sign"]]
    colors = [c if p < 0.05 else "#cccccc" for c, p in zip(colors, sub["p"])]

    x = np.arange(len(sub))
    ax.bar(x, sub["mean"], color=colors, edgecolor="black", linewidth=0.8,
           yerr=sub["ci"], capsize=5, error_kw={"linewidth": 1.2})
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    _add_sig_labels(ax, x, sub["mean"].values, sub["p"].values)
    ax.set_xticks(x); ax.set_xticklabels(sub["target"], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("ΔP(MYE)")
    ax.set_title(PHASE_LABELS[phase])
    ax.grid(axis="y", alpha=0.3)

from matplotlib.patches import Patch
legend_els = [Patch(facecolor="#D65F5F", label="Sustain (+)"),
              Patch(facecolor="#4878CF", label="Brake (−)"),
              Patch(facecolor="#cccccc", label="ns")]
axes[0].legend(handles=legend_els, fontsize=9)
fig.suptitle("Interventional XRO — Mode-Specific Causal Effect", fontsize=13)
fig.tight_layout()
plt.show()

---
## 7. Seasonality of Control

When in the seasonal cycle does the agent act most strongly? Highlights the spring predictability barrier (Mar–May).

In [ ]:
seas_dir = BASE / "seasonality"
d_seas   = _load_npz(seas_dir / "seasonality_of_control.npz")

month_names   = list(d_seas["months"])      # ['Jan', ..., 'Dec']
var_names     = list(d_seas["var_names"])    # includes Nino3.4; drivers are [1:]
drivers       = var_names[1:]
abs_by_month  = d_seas["abs_by_month"]       # [12, n_modes]
abs_sem       = d_seas["abs_sem"]
signed_by_month = d_seas["signed_by_month"]  # [12, n_modes]
total_by_month = d_seas["total_by_month"]    # [12]
spring_idx    = list(d_seas["spring_barrier"])  # [2, 3, 4]

spring_frac = total_by_month[spring_idx].sum() / total_by_month.sum()
print(f"Spring (Mar-May) share: {spring_frac*100:.1f}%  (even = 25.0%)")

In [ ]:
# --- 7a. Month × mode heatmap (absolute forcing) ---
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(abs_by_month.T, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(12)); ax.set_xticklabels(month_names)
ax.set_yticks(range(len(drivers))); ax.set_yticklabels(drivers)
ax.set_title("Mean |Scaled Forcing| by Calendar Month × Mode")
plt.colorbar(im, ax=ax, label="Mean |forcing|")
# Highlight spring
for m in spring_idx:
    ax.axvline(m - 0.5, color="cyan", linewidth=2)
    ax.axvline(m + 0.5, color="cyan", linewidth=2)
fig.tight_layout()
plt.show()

In [ ]:
# --- 7b. Monthly total forcing bar ---
fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ["#E57373" if m in spring_idx else "#90CAF9" for m in range(12)]
ax.bar(range(12), total_by_month, color=bar_colors, edgecolor="black", linewidth=0.8)
ax.set_xticks(range(12)); ax.set_xticklabels(month_names)
ax.set_ylabel("Total |scaled forcing| (sum over modes)")
ax.set_title(f"Monthly Forcing Total  |  Spring share = {spring_frac*100:.1f}%")
ax.grid(axis="y", alpha=0.3)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor="#E57373", label="Spring (Mar-May)"),
                   Patch(facecolor="#90CAF9", label="Other months")], fontsize=9)
fig.tight_layout()
plt.show()

---
## 8. Precursor Composite

Mean XRO state in the months preceding spontaneous multi-year ENSO events in a free-running emulator (no agent). Validates agent-discovered drivers against the emulator's own natural precursor patterns.

In [ ]:
prec_dir = BASE / "precursor_composite"
d_prec   = _load_npz(prec_dir / "precursor_composite.npz")

prec_vars = list(d_prec["var_names"])  # 10 modes
lead      = int(d_prec["lead"])
lag_months = np.arange(-lead, 0)

for p in PHASES:
    n = int(d_prec[f"{p}_n"])
    print(f"  {p:8s}: {n} events")

In [ ]:
# --- 8a. Precursor composite trajectories per phase (top 5 modes) ---
fig, axes = plt.subplots(1, len(PHASES), figsize=(15, 5), sharey=False)
top_n = 5

for ax, phase in zip(axes, PHASES):
    n = int(d_prec[f"{phase}_n"])
    if n == 0:
        ax.text(0.5, 0.5, "No events", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(PHASE_LABELS[phase]); continue

    traj_mean = d_prec[f"{phase}_traj_mean"]  # [lead, n_modes]
    traj_ci   = d_prec[f"{phase}_traj_ci"]
    pat_mean  = d_prec[f"{phase}_pattern_mean"]

    # Pick top modes by absolute average precursor signal
    top_idx = np.argsort(np.abs(pat_mean))[::-1][:top_n]
    cmap = plt.get_cmap("tab10")
    for k, mi in enumerate(top_idx):
        col = cmap(k)
        y = traj_mean[:, mi]
        ci = traj_ci[:, mi]
        ax.plot(lag_months, y, color=col, label=prec_vars[mi], linewidth=2)
        ax.fill_between(lag_months, y - ci, y + ci, color=col, alpha=0.15)

    ax.axhline(0, color="grey", linewidth=0.6, linestyle="--")
    ax.axvline(0, color="red", linewidth=1.0, linestyle=":", label="Onset")
    ax.set_xlabel("Months before onset")
    ax.set_ylabel("Normalised state")
    ax.set_title(f"{PHASE_LABELS[phase]} (N={n})")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.suptitle("Spontaneous-MYE Precursor Composite (XRO, no agent)", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# --- 8b. Precursor pattern bar (lead-averaged) ---
fig, axes = plt.subplots(1, len(PHASES), figsize=(15, 5), sharey=False)
for ax, phase in zip(axes, PHASES):
    n = int(d_prec[f"{phase}_n"])
    if n == 0:
        ax.text(0.5, 0.5, "No events", ha="center", va="center", transform=ax.transAxes); continue
    pat_mean = d_prec[f"{phase}_pattern_mean"]
    pat_ci   = d_prec[f"{phase}_pattern_ci"]
    order = np.argsort(pat_mean)[::-1]
    x = np.arange(len(prec_vars))
    colors = [PHASE_COLORS[phase] if abs(pat_mean[i]) > pat_ci[i] else "#cccccc" for i in order]
    ax.bar(x, pat_mean[order], color=colors, edgecolor="black", linewidth=0.8,
           yerr=pat_ci[order], capsize=5, error_kw={"linewidth": 1.2})
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_xticks(x); ax.set_xticklabels([prec_vars[i] for i in order], rotation=45, ha="right")
    ax.set_ylabel("Mean precursor amplitude")
    ax.set_title(f"{PHASE_LABELS[phase]} (N={n})")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Precursor Pattern (Lead-Averaged Mean State)", fontsize=13)
fig.tight_layout()
plt.show()

---
## 9. Mutual Information

MI(action_i, future ENSO class at lag τ). Identifies which forcing dimensions carry the most information about upcoming ENSO state.

> Run `uv run scripts/analysis/mutual_information.py --model ensemble` to generate `mi_results.npz`.

In [ ]:
mi_path = BASE / "mutual_information" / "mi_results.npz"
if not mi_path.exists():
    print(f"[skip] {mi_path} not found. Run mutual_information.py first.")
else:
    d_mi = _load_npz(mi_path)
    mi_primary_per_run = d_mi["mi_primary_per_run"]  # [n_runs, n_actions]
    mean_lag_profiles  = d_mi["mean_lag_profiles"]    # [n_lags, n_actions]
    lags               = list(d_mi["lags"])
    action_names       = list(d_mi["action_names"])

    mi_mean = mi_primary_per_run.mean(axis=0)
    mi_std  = mi_primary_per_run.std(axis=0, ddof=1)
    mi_pvals = np.array([ttest_1samp(mi_primary_per_run[:, i], 0).pvalue
                         for i in range(mi_primary_per_run.shape[1])])

    print(f"Primary lag = {lags[0]} months  |  {len(action_names)} actions  |  {len(lags)} lags")
    order = np.argsort(mi_mean)[::-1]
    for i in order:
        print(f"  {action_names[i]:<8}  MI={mi_mean[i]*1000:.3f} mn  p={mi_pvals[i]:.4f}  {_sig_stars(mi_pvals[i])}")

In [ ]:
if mi_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Ranked MI bar
    ax = axes[0]
    order = np.argsort(mi_mean)[::-1]
    n_act = len(action_names)
    x = np.arange(n_act)
    ci95 = 1.96 * mi_std / np.sqrt(mi_primary_per_run.shape[0])
    bar_colors = ["#4878CF" if mi_pvals[i] < 0.05 else "#cccccc" for i in order]
    ax.bar(x, mi_mean[order] * 1000, color=bar_colors, edgecolor="black", linewidth=0.8,
           yerr=ci95[order] * 1000, capsize=5, error_kw={"linewidth": 1.2})
    _add_sig_labels(ax, x, mi_mean[order] * 1000, mi_pvals[order])
    ax.set_xticks(x); ax.set_xticklabels([action_names[i] for i in order], rotation=45, ha="right")
    ax.set_ylabel("MI (millinats)")
    ax.set_title(f"MI(action, ENSO class) at lag={lags[0]} months")
    ax.grid(axis="y", alpha=0.3)

    # MI vs lag heatmap
    ax = axes[1]
    im = ax.imshow(mean_lag_profiles.T * 1000, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(lags))); ax.set_xticklabels([f"{l}mo" for l in lags])
    ax.set_yticks(range(n_act)); ax.set_yticklabels(action_names)
    ax.set_xlabel("Lag"); ax.set_title("MI vs Lag (millinats)")
    plt.colorbar(im, ax=ax)

    fig.tight_layout()
    plt.show()

---
## 10. Integrated Gradients

Axiomatic attribution — IG sums exactly to the output difference (completeness). Shows which observations the policy and value function attend to.

> Run `uv run scripts/analysis/integrated_gradients.py --model ensemble` to generate `ig_results.npz`.

In [ ]:
ig_path = BASE / "integrated_gradients" / "ig_results.npz"
if not ig_path.exists():
    print(f"[skip] {ig_path} not found. Run integrated_gradients.py first.")
else:
    d_ig = _load_npz(ig_path)
    policy_imp = d_ig["policy_importance_per_run"]  # [n_runs, n_obs]
    value_imp  = d_ig["value_importance_per_run"]
    obs_names  = list(d_ig["obs_names"])
    n_runs_ig  = int(d_ig["n_runs"])

    pol_mean = policy_imp.mean(axis=0)
    val_mean = value_imp.mean(axis=0)
    pol_pvals = np.array([ttest_1samp(policy_imp[:, i], 0).pvalue for i in range(policy_imp.shape[1])])
    val_pvals = np.array([ttest_1samp(value_imp[:, i],  0).pvalue for i in range(value_imp.shape[1])])

    print(f"N_runs={n_runs_ig}  |  {len(obs_names)} obs dimensions")

In [ ]:
if ig_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, (mean_v, pvals_v, title) in zip(axes, [
        (pol_mean, pol_pvals, "Policy IG Attribution  (Σ|IG| across actions)"),
        (val_mean, val_pvals, "Value IG Attribution  (|IG(V, obs)|)"),
    ]):
        order = np.argsort(mean_v)[::-1]
        x = np.arange(len(obs_names))
        ci95 = 1.96 * policy_imp.std(axis=0, ddof=1) / np.sqrt(n_runs_ig)
        colors = ["#4878CF" if pvals_v[i] < 0.05 else "#cccccc" for i in order]
        ax.bar(x, mean_v[order], color=colors, edgecolor="black", linewidth=0.8,
               yerr=ci95[order], capsize=5, error_kw={"linewidth": 1.2})
        _add_sig_labels(ax, x, mean_v[order], pvals_v[order])
        ax.set_xticks(x); ax.set_xticklabels([obs_names[i] for i in order], rotation=45, ha="right")
        ax.set_title(title); ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Integrated Gradients", fontsize=13)
    fig.tight_layout()
    plt.show()

---
## 11. Gradient Sensitivity

Policy Jacobian (∂action/∂obs) and value gradient (∂V/∂obs) — how much each observation dimension moves the policy output or value estimate.

> Run `uv run scripts/analysis/gradient_sensitivity.py --model ensemble` to generate `gradient_results.npz`.

In [ ]:
gs_path = BASE / "gradient_sensitivity" / "gradient_results.npz"
if not gs_path.exists():
    print(f"[skip] {gs_path} not found. Run gradient_sensitivity.py first.")
else:
    d_gs = _load_npz(gs_path)
    gs_policy = d_gs["policy_importance_per_run"]  # [n_runs, n_obs]
    gs_value  = d_gs["value_importance_per_run"]
    gs_jac    = d_gs["mean_jacobian_all"]            # [n_actions, n_obs]
    gs_obs    = list(d_gs["obs_names"])
    gs_act    = list(d_gs["action_names"])
    n_runs_gs = int(d_gs["n_runs"])

    gs_pol_mean  = gs_policy.mean(axis=0)
    gs_val_mean  = gs_value.mean(axis=0)
    gs_pol_pvals = np.array([ttest_1samp(gs_policy[:, i], 0).pvalue for i in range(gs_policy.shape[1])])
    gs_val_pvals = np.array([ttest_1samp(gs_value[:, i],  0).pvalue for i in range(gs_value.shape[1])])
    print(f"N_runs={n_runs_gs}  |  obs: {len(gs_obs)}  |  actions: {len(gs_act)}")

In [ ]:
if gs_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, (mean_v, pvals_v, label) in zip(axes, [
        (gs_pol_mean, gs_pol_pvals, "Policy Sensitivity  (Σ|∂a/∂o|)"),
        (gs_val_mean, gs_val_pvals, "Value Sensitivity  (|∂V/∂o|)"),
    ]):
        order = np.argsort(mean_v)[::-1]
        x = np.arange(len(gs_obs))
        ci95 = 1.96 * gs_policy.std(axis=0, ddof=1) / np.sqrt(n_runs_gs)
        colors = ["#4878CF" if pvals_v[i] < 0.05 else "#cccccc" for i in order]
        ax.bar(x, mean_v[order], color=colors, edgecolor="black", linewidth=0.8,
               yerr=ci95[order], capsize=5, error_kw={"linewidth": 1.2})
        _add_sig_labels(ax, x, mean_v[order], pvals_v[order])
        ax.set_xticks(x); ax.set_xticklabels([gs_obs[i] for i in order], rotation=45, ha="right")
        ax.set_title(label); ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Gradient-Based Sensitivity", fontsize=13)
    fig.tight_layout()
    plt.show()

In [ ]:
if gs_path.exists():
    # Policy Jacobian heatmap |∂action/∂obs|
    fig, ax = plt.subplots(figsize=(12, 5))
    im = ax.imshow(gs_jac[:, :len(gs_obs)], cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(gs_obs))); ax.set_xticklabels(gs_obs, rotation=45, ha="right")
    ax.set_yticks(range(len(gs_act))); ax.set_yticklabels(gs_act)
    ax.set_xlabel("Observation"); ax.set_ylabel("Action")
    ax.set_title("Policy Jacobian: Mean |∂action/∂observation|")
    plt.colorbar(im, ax=ax)

    for i in range(gs_jac.shape[0]):
        for j in range(min(gs_jac.shape[1], len(gs_obs))):
            val = gs_jac[i, j]
            color = "white" if val > gs_jac.max() * 0.6 else "black"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=7, color=color)

    fig.tight_layout()
    plt.show()